# CocoAnalytica: External F1 Score and Confusion Matrix

This notebook performs the external-only evaluation for the TensorFlow Lite artifact used by the website. It contains no training, validation, or internal held-out `test` images. The training and per-image test-folder score workflow remains in `model.ipynb`.

## What this report measures

Only approved, independently sourced photos in `external_test_set` are evaluated. The audit requires provenance and rejects exact pixel duplicates of the prepared training, validation, and internal test dataset.

This is not yet a 20-per-class or 50-per-class external benchmark: it currently contains 12 Healthy, 12 Yellowing, 5 Coconut Scale Insect, and 2 Rhinoceros Beetle images. A 20-per-class table would require adding more unique approved external photographs; duplicating `Y04` would not be valid.

In [ ]:
from __future__ import annotations
import json
import sys
from pathlib import Path

from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / 'backend/app/ml/predictor.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))
sys.path.insert(0, str(PROJECT_ROOT / 'backend/scripts'))
from notebook_workflow import CLASSES, evaluate_api, external_test_status

MODEL_DIR = PROJECT_ROOT / 'Thesis AI Model'
OUTPUT_DIR = MODEL_DIR / 'model_outputs'
EXTERNAL_TEST_DIR = MODEL_DIR / 'external_test_set'
REFERENCE_DATASET_DIR = MODEL_DIR / 'notebook_dataset' / '4e327ceeeb11'
TFLITE_PATH = OUTPUT_DIR / 'coconut_leaf_multilabel_cnn.tflite'
LABEL_CONFIG_PATH = OUTPUT_DIR / 'label_config.json'
EXPECTED_EXTERNAL_IMAGES_PER_CLASS = 50

In [ ]:
def display_f1_score_table(report, title):
    lines = [
        f'### F1 Score Table - {title}',
        '',
        '| Condition | Precision | Recall | F1 score | Images |',
        '| --- | ---: | ---: | ---: | ---: |',
    ]
    for label in CLASSES:
        row = report['per_class'][label]
        lines.append(
            f"| {label.replace('_', ' ')} | {row['precision']:.1%} | {row['recall']:.1%} | "
            f"**{row['f1']:.1%}** | {row['support']} |"
        )
    lines.extend([
        '',
        '| Overall measure | Score |',
        '| --- | ---: |',
        f"| Accuracy | **{report['accuracy']:.1%}** |",
        f"| Macro F1 | **{report['macro_f1']:.1%}** |",
        f"| Weighted F1 | **{report['weighted_f1']:.1%}** |",
    ])
    display(Markdown('\n'.join(lines)))


def display_confusion_matrix(report, title):
    lines = [
        f'### Confusion Matrix - {title}',
        'Rows are actual conditions; columns are website-model predictions.',
        '',
        '| Actual / Predicted | Healthy | Yellowing | Coconut Scale Insect | Rhinoceros Beetle |',
        '| --- | ---: | ---: | ---: | ---: |',
    ]
    for label, row in zip(CLASSES, report['confusion_matrix']):
        lines.append(f"| {label.replace('_', ' ')} | " + ' | '.join(map(str, row)) + ' |')
    display(Markdown('\n'.join(lines)))

In [ ]:
external_status = external_test_status(
    EXTERNAL_TEST_DIR,
    expected_per_class=EXPECTED_EXTERNAL_IMAGES_PER_CLASS,
    reference_dataset=REFERENCE_DATASET_DIR,
)
print(json.dumps(external_status, indent=2))

if not external_status['audited']:
    raise RuntimeError('External evaluation stopped: resolve the audit findings first.')

external_report = evaluate_api(
    TFLITE_PATH,
    LABEL_CONFIG_PATH,
    EXTERNAL_TEST_DIR,
    split=None,
    runtime='tflite',
)
external_report.update(
    benchmark='provisional independently sourced external originals',
    benchmark_status='provisional: fewer than 50 reviewed images per class',
    images_per_class=external_status['counts'],
    expected_images_per_class=EXPECTED_EXTERNAL_IMAGES_PER_CLASS,
    provenance_file=str(EXTERNAL_TEST_DIR / 'provenance.json'),
)
(OUTPUT_DIR / 'external_test_provisional_evaluation.json').write_text(
    json.dumps(external_report, indent=2) + '\n', encoding='utf-8'
)
display_f1_score_table(external_report, 'Approved external images only (TensorFlow Lite website runtime)')
display_confusion_matrix(external_report, 'Approved external images only')
print('Saved:', OUTPUT_DIR / 'external_test_provisional_evaluation.json')

### F1 Score Table - Approved External Images Only

| Condition | Precision | Recall | F1 score | Images |
| --- | ---: | ---: | ---: | ---: |
| Healthy | 30.0% | 25.0% | **27.3%** | 12 |
| Yellowing | 33.3% | 8.3% | **13.3%** | 12 |
| Coconut Scale Insect | 100.0% | 20.0% | **33.3%** | 5 |
| Rhinoceros Beetle | 0.0% | 0.0% | **0.0%** | 2 |

| Overall measure | Score |
| --- | ---: |
| Accuracy | **16.1%** |
| Macro F1 | **18.5%** |
| Weighted F1 | **21.1%** |

### Confusion Matrix - Approved External Images Only
Rows are actual conditions; columns are website-model predictions.

| Actual / Predicted | Healthy | Yellowing | Coconut Scale Insect | Rhinoceros Beetle |
| --- | ---: | ---: | ---: | ---: |
| Healthy | 3 | 2 | 0 | 7 |
| Yellowing | 5 | 1 | 0 | 6 |
| Coconut Scale Insect | 0 | 0 | 1 | 4 |
| Rhinoceros Beetle | 2 | 0 | 0 | 0 |